# Agent 1 - precision and recall

Compares Agent 1's output (rules + llama3:8b) on the 20 hand-labelled test documents with
the gold annotations.

* **Recall**: of all the sensitive items in the documents, how many Agent 1 found.
  The primary figure, because a missed item is a privacy leak.
* **Precision**: of everything Agent 1 flagged, how much was really sensitive.
* **F1**: the balance of the two.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`. Nothing is stored in your
Google Drive.

1. **Cell ① Start**: asks for `3rd_sem_el_demo.zip`. With `RUN_LIVE` unticked it only
   installs the scorer (about 1 minute) and cell ② shows the saved run of 18 Oct 2026
   instantly. Tick `RUN_LIVE` to download the model and run Agent 1 again.
2. **Cell ②**: shows the scores.

In [ ]:
#@title ① Start - run once per session
#@markdown Untick to show the saved run of 18 Oct 2026 instantly (about 1 minute of setup,
#@markdown no model download). Ticked: downloads the model and runs Agent 1 again (3-5 min
#@markdown setup, then 1-2 min for the 20 documents).
RUN_LIVE = False  #@param {type:"boolean"}

import os
import sys
import time
import zipfile
from pathlib import Path

t_start = time.time()
REPO = Path("/content/3rd_sem_el")
if not REPO.exists():
    from google.colab import files

    print("Choose the code zip: 3rd_sem_el_demo.zip")
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")

sys.path.insert(0, str(REPO / "tools"))
import colab_cache

colab_cache.setup(REPO, model="llama3:8b" if RUN_LIVE else None)

os.environ["SANITISE_LLM_BACKEND"] = "ollama"
os.environ["SANITISE_MODEL"] = "llama3:8b"
os.environ["SANITISE_LLM_TIMEOUT_S"] = "300"

from IPython.display import HTML, display

print(f"ready in {time.time() - t_start:.0f} s - now run cell 2")

In [ ]:
#@title ② Precision and recall on the 20 gold documents
from sanitise.evaluation import scoring
from sanitise.results import load_gold, load_predictions
from sanitise.showcase import dataset_summary, render_scores, run_fixtures_sync

FIXTURES, GOLD = REPO / "data" / "fixtures", REPO / "data" / "gold"

if RUN_LIVE:
    colab_cache.install_ollama_and_model("llama3:8b")  # returns at once if already set up
    PRED = Path("/content/predictions_live")
    t = time.time()
    results = run_fixtures_sync(FIXTURES, PRED)
    print(f"ran Agent 1 on {len(results)} documents in {time.time() - t:.0f} s")
    NOTE = None
else:
    # The saved run is scored against the gold set as it was when the run was made.
    PRED = REPO / "data" / "predictions" / "llama3_8b"
    GOLD = REPO / "data" / "predictions" / "llama3_8b_gold"
    NOTE = (
        "Saved run of 18 Oct 2026 (Colab T4), scored against the gold set of that date "
        "(before the SYMPTOM category). Tick RUN_LIVE in cell 1 for the current version."
    )
    print("showing the saved run of 18 Oct 2026 - tick RUN_LIVE in cell 1 to rerun")

metrics = scoring.score(load_predictions(PRED), load_gold(GOLD))
display(HTML(render_scores(metrics, dataset_summary(FIXTURES, GOLD), note=NOTE)))